# Import and Model Loading:

In [1]:
import sys
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from xray.interpretability import *

c:\Users\megha\Desktop\xray-transformers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#IMPORTS:

import torch
import matplotlib.pyplot as plt

from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer
)

from xray.interpretability import DistilBertInterpreter

# LOAD THE HUGGINGFACE MODEL:

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
).to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
)

print("Model loaded!")

Device: cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11666.42it/s]


Model loaded!


In [12]:
# CREATE THE INTERPRETER:

interpreter = DistilBertInterpreter(
    model,
    tokenizer,
    device
)

# Test Cases:

In [4]:
test_texts = [
    "This movie was absolutely fantastic and I loved every minute of it.",
    "This movie was terrible, boring, and completely disappointing.",
    "This was not a bad movie at all.",
    "The acting was excellent, but the story was painfully boring.",
    "I expected this movie to be terrible, but it was actually wonderful.",
    "What a fantastic movie... if you enjoy falling asleep."
]

for i, text in enumerate(test_texts, 1):
    print(f"{i}. {text}")

1. This movie was absolutely fantastic and I loved every minute of it.
2. This movie was terrible, boring, and completely disappointing.
3. This was not a bad movie at all.
4. The acting was excellent, but the story was painfully boring.
5. I expected this movie to be terrible, but it was actually wonderful.
6. What a fantastic movie... if you enjoy falling asleep.


# Original Predictions:

In [5]:
def predict_text(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)

    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0, predicted_class].item()

    return predicted_class, confidence

In [6]:
original_results = []

for text in test_texts:
    prediction, confidence = predict_text(text)

    original_results.append({
        "text": text,
        "prediction": prediction,
        "label": "POSITIVE" if prediction == 1 else "NEGATIVE",
        "confidence": confidence
    })

original_df = pd.DataFrame(original_results)

original_df

,text,prediction,label,confidence
0,This movie was absolutely fantastic and I love...,1,POSITIVE,0.995928
1,"This movie was terrible, boring, and completel...",0,NEGATIVE,0.997209
2,This was not a bad movie at all.,1,POSITIVE,0.825081
3,"The acting was excellent, but the story was pa...",0,NEGATIVE,0.978495
4,"I expected this movie to be terrible, but it w...",1,POSITIVE,0.986044
5,What a fantastic movie... if you enjoy falling...,1,POSITIVE,0.989481


# Word Removal Perturbation:

In [21]:
text = test_texts[0]

words = text.split()

original_prediction, original_confidence = predict_text(text)

perturbation_results = []

for i, word in enumerate(words):
    perturbed_words = words[:i] + words[i+1:]
    perturbed_text = " ".join(perturbed_words)

    prediction, confidence = predict_text(perturbed_text)

    perturbation_results.append({
        "removed_word": word,
        "perturbed_text": perturbed_text,
        "prediction": prediction,
        "confidence": confidence,
        "confidence_change": confidence - original_confidence
    })

perturbation_df = pd.DataFrame(perturbation_results)

perturbation_df.head()

,removed_word,perturbed_text,prediction,confidence,confidence_change
0,This,movie was absolutely fantastic and I loved eve...,1,0.995387,-0.000540
1,movie,This was absolutely fantastic and I loved ever...,1,0.995328,-0.000599
2,was,This movie absolutely fantastic and I loved ev...,1,0.996060,0.000132
3,absolutely,This movie was fantastic and I loved every min...,1,0.996027,0.000100
4,fantastic,This movie was absolutely and I loved every mi...,1,0.995801,-0.000127


# Get attribution:

In [10]:
def get_word_attributions(text):
    """
    Generate word-level Integrated Gradients
    for a given input sentence.
    """

    (
        tokens,
        attributions,
        delta,
        input_output,
        baseline_output,
        total_attribution,
        completeness_error
    ) = interpreter.attribute(text)

    words, scores = interpreter.aggregate_tokens(
        tokens,
        attributions
    )

    return words, scores

# Align IG Attribution with deletion effect:
Do words receiving high IG attribution actually have a larger effect when removed?

Measure attribution/deletion correlation:

In [20]:
from scipy.stats import spearmanr
# Spearman correlation because we're interested in whether words ranked highly 
# by IG also tend to have larger deletion effects.

In [18]:
def evaluate_deletion_faithfulness(text):
    # Get IG word-level attributions
    words, attributions = get_word_attributions(text)

    # Original prediction
    original_prediction, original_confidence = predict_text(text)

    results = []

    for i, word in enumerate(words):
        # Remove exactly one word
        perturbed_words = words[:i] + words[i+1:]
        perturbed_text = " ".join(perturbed_words)

        prediction, confidence = predict_text(perturbed_text)

        results.append({
            "word": word,
            "attribution": float(attributions[i]),
            "deletion_effect": original_confidence - confidence
        })

    df = pd.DataFrame(results)

    # Spearman correlation between IG ranking and deletion effect
    correlation, p_value = spearmanr(
        df["attribution"],
        df["deletion_effect"]
    )

    return df, correlation, p_value

In [19]:
comparison_df, correlation, p_value = evaluate_deletion_faithfulness(
    test_texts[0]
)

print(f"Spearman correlation: {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

comparison_df

Spearman correlation: 0.4011
P-value: 0.1744


,word,attribution,deletion_effect
0,this,0.217276,0.000540
1,movie,0.148403,0.000599
2,was,0.045538,-0.000132
3,absolutely,0.234302,-0.000100
4,fantastic,0.472755,0.000127
5,and,0.427234,0.000259
6,i,0.217258,0.000118
7,loved,0.485604,0.000914
8,every,0.038579,0.000231
9,minute,0.014611,-0.000164


# Run faithfulness evaluation across all test cases:

In [22]:
faithfulness_results = []

for text in test_texts:
    comparison_df, correlation, p_value = evaluate_deletion_faithfulness(text)

    prediction, confidence = predict_text(text)

    faithfulness_results.append({
        "text": text,
        "prediction": "POSITIVE" if prediction == 1 else "NEGATIVE",
        "confidence": confidence,
        "spearman_correlation": correlation,
        "p_value": p_value,
        "num_words": len(comparison_df)
    })

faithfulness_df = pd.DataFrame(faithfulness_results)

faithfulness_df

,text,prediction,confidence,spearman_correlation,p_value,num_words
0,This movie was absolutely fantastic and I love...,POSITIVE,0.995928,0.401099,0.174357,13
1,"This movie was terrible, boring, and completel...",NEGATIVE,0.997209,0.481818,0.133434,11
2,This was not a bad movie at all.,POSITIVE,0.825081,0.133333,0.732368,9
3,"The acting was excellent, but the story was pa...",NEGATIVE,0.978495,0.552448,0.062511,12
4,"I expected this movie to be terrible, but it w...",POSITIVE,0.986044,0.468132,0.091376,14
5,What a fantastic movie... if you enjoy falling...,POSITIVE,0.989481,0.850842,0.000227,13


# Target class logit prediction:

Do words that IG considers important also cause a larger decrease in the model's target-class logit when deleted?

In [23]:
def predict_target_logit(text, target_class):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.logits[0, target_class].item()

In [24]:
def evaluate_logit_faithfulness(text):
    # Get IG word-level attributions
    words, attributions = get_word_attributions(text)

    # Get original prediction
    original_prediction, _ = predict_text(text)

    # Original target-class logit
    original_logit = predict_target_logit(
        text,
        original_prediction
    )

    results = []

    for i, word in enumerate(words):
        # Remove exactly one word
        perturbed_words = words[:i] + words[i+1:]
        perturbed_text = " ".join(perturbed_words)

        # Target-class logit after deletion
        perturbed_logit = predict_target_logit(
            perturbed_text,
            original_prediction
        )

        # Positive value means deleting the word reduced
        # the target-class logit.
        deletion_effect = original_logit - perturbed_logit

        results.append({
            "word": word,
            "attribution": float(attributions[i]),
            "deletion_effect": deletion_effect
        })

    df = pd.DataFrame(results)

    correlation, p_value = spearmanr(
        df["attribution"],
        df["deletion_effect"]
    )

    return df, correlation, p_value

In [25]:
logit_comparison_df, correlation, p_value = evaluate_logit_faithfulness(
    test_texts[0]
)

print(f"Spearman correlation: {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

logit_comparison_df

Spearman correlation: 0.4396
P-value: 0.1329


,word,attribution,deletion_effect
0,this,0.217276,0.062541
1,movie,0.148403,0.073895
2,was,0.045538,-0.010418
3,absolutely,0.234302,-0.011374
4,fantastic,0.472755,0.033403
5,and,0.427234,0.034086
6,i,0.217258,0.017441
7,loved,0.485604,0.107415
8,every,0.038579,-0.001210
9,minute,0.014611,-0.029291


In [26]:
logit_faithfulness_results = []

for text in test_texts:
    comparison_df, correlation, p_value = evaluate_logit_faithfulness(text)

    prediction, confidence = predict_text(text)

    logit_faithfulness_results.append({
        "text": text,
        "prediction": "POSITIVE" if prediction == 1 else "NEGATIVE",
        "confidence": confidence,
        "spearman_correlation": correlation,
        "p_value": p_value,
        "num_words": len(comparison_df)
    })

logit_faithfulness_df = pd.DataFrame(logit_faithfulness_results)

logit_faithfulness_df

,text,prediction,confidence,spearman_correlation,p_value,num_words
0,This movie was absolutely fantastic and I love...,POSITIVE,0.995928,0.439560,0.132860,13
1,"This movie was terrible, boring, and completel...",NEGATIVE,0.997209,0.481818,0.133434,11
2,This was not a bad movie at all.,POSITIVE,0.825081,0.200000,0.605901,9
3,"The acting was excellent, but the story was pa...",NEGATIVE,0.978495,0.552448,0.062511,12
4,"I expected this movie to be terrible, but it w...",POSITIVE,0.986044,0.551648,0.040849,14
5,What a fantastic movie... if you enjoy falling...,POSITIVE,0.989481,0.845317,0.000274,13
